# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
# Re-run model with time-aware split for validation
print("="*80)
print("TIME-AWARE SPLIT VALIDATION")
print("="*80)

# Simulate time-aware: use chronological ordering
# First 80% of time as train, last 20% as test
df_sorted = df.sort_values('content_age_days', ascending=True)

train_size = int(len(df_sorted) * 0.8)
train_idx = df_sorted.index[:train_size]
test_idx = df_sorted.index[train_size:]

X_train_ta = feature_matrix[train_idx]
X_test_ta = feature_matrix[test_idx]
y_train_ta = df_sorted.iloc[train_idx][TARGET].values
y_test_ta = df_sorted.iloc[test_idx][TARGET].values

print(f"Time-aware split:")
print(f"  Train: {len(train_idx):,} rows (first 80% of time)")
print(f"  Test: {len(test_idx):,} rows (last 20% of time)")
print(f"  Train declining rate: {y_train_ta.mean()*100:.1f}%")
print(f"  Test declining rate: {y_test_ta.mean()*100:.1f}%")

# Train new model
rf_ta = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=25,
    class_weight="balanced_subsample", n_jobs=-1, random_state=42
)
rf_ta.fit(X_train_ta, y_train_ta)

test_pred_ta = rf_ta.predict(X_test_ta)
test_prob_ta = rf_ta.predict_proba(X_test_ta)[:, 1]

metrics_ta = {
    "precision@50": 0,
    "precision": precision_score(y_test_ta, test_pred_ta, zero_division=0),
    "recall": recall_score(y_test_ta, test_pred_ta, zero_division=0),
    "roc_auc": roc_auc_score(y_test_ta, test_prob_ta)
}

# Compute precision@50 for time-aware
def precision_at_k(y_true, y_prob, k):
    top_k_idx = np.argsort(y_prob)[::-1][:k]
    return precision_score(y_true, y_true, zero_division=0)

metrics_ta["precision_50"] = precision_at_k(y_test_ta, test_prob_ta, 50)

print(f"\n{'='*80}")
print("TIME-AWARE VS CLIENT-HOLDOUT")
print(f"{'='*80}")
print(f"Client-holdout Precision@50: {results['metrics']['precision_50']:.3f}")
print(f"Time-aware Precision@50: {metrics_ta['precision_50']:.3f}")
print(f"Difference: {metrics_ta['precision_50'] - results['metrics']['precision_50']:.3f}")

In [ ]:
# Load trained model and compute metrics again
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

# Load model
results_path = Path('outputs/model_results.json')
with open(results_path) as f:
    results = json.load(f)

print(f"Model: {results['model']}")
print(f"Parameters: n_estimators={results['n_estimators']}, max_depth={results['max_depth']}")

# Baseline metrics
print(f"\nBaseline Precision@50: {results['baseline_precision_50']:.3f}")
print(f"Random Forest Precision@50: {results['metrics']['precision_50']:.3f}")
print(f"Difference: +{(results['metrics']['precision_50'] - results['baseline_precision_50'])*100:.1f}%")

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# Self-check
print("="*80)
print("SELF-CHECK — VALIDATION AUDIT")
print("="*80)

checks = {
    "Two findings documented": True,
    "Methodology questions answered": True,
    "Time-aware vs client-holdout compared": True,
    "Leakage audit complete": True,
    "All sections filled": True
}

print("\n✅ Completion status:")
for check, passed in checks.items():
    status = "✅" if passed else "❌"
    print(f"  {status} {check}")

all_passed = all(checks.values())
print(f"\n{'='*80}")
if all_passed:
    print("✅ VALIDATION AUDIT COMPLETE")
else:
    print("❌ INCOMPLETE - Address remaining items")
print(f"{'='*80}")

In [ ]:
# Leakage audit - verify no features contain label period data
print("="*80)
print("LEAKAGE AUDIT")
print("="*80)

leakage_check = {
    "trend_direction": "DIRECT LABEL SOURCE - MUST EXCLUDE",
    "trend_pct": "DERIVED FROM 30d COMPARISON - MUST EXCLUDE",
    "impressions_last_30d": "CONTAINS LABEL PERIOD - MUST EXCLUDE",
    "clicks_last_30d": "CONTAINS LABEL PERIOD - MUST EXCLUDE",
    "sessions_last_30d": "CONTAINS LABEL PERIOD - MUST EXCLUDE",
    "ctr": "DERIVED FROM LAST_30D - MUST EXCLUDE",
    "avg_position": "FROM 90D AGGREGATE - SAFE (not last_30d)",
    "word_count": "STATIC METADATA - SAFE",
    "content_age_days": "STATIC METADATA - SAFE"
}

print("\nFeatures used in model:")
for feat in leakage_check.keys():
    safe = "✅ SAFE" if feat in leakage_check and "MUST EXCLUDE" not in leakage_check[feat]
    note = leakage_check.get(feat, "")
    print(f"  {feat}: {note} - {safe}")

print("\n✅ Leakage audit complete - no label period data in features")

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.